In [ ]:
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

## 10. Advanced Model Training

Train LightGBM, XGBoost, and CatBoost with cross-validation.
Compare feature importance across backends.

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)
print(f"Data loaded: {train.shape}")
print(f"Fraud rate : {train['isFraud'].mean()*100:.2f}%")

### 10.1 LightGBM

In [ ]:
# 5-fold cross validation dengan stratifikasi
exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_scores = []
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "num_leaves": 31,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 42
}

start = time.perf_counter()
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=200,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    
    preds = model.predict(X_va, num_iteration=model.best_iteration)
    lgb_scores.append(roc_auc_score(y_va, preds))

elapsed_lgb = time.perf_counter() - start
mean_lgb = np.mean(lgb_scores)
std_lgb = np.std(lgb_scores)
print(f"Mean AUC: {mean_lgb:.5f} \u00b1 {std_lgb:.5f}")
print(f"Fold scores: {[round(s, 5) for s in lgb_scores]}")

In [ ]:
# latih model final dengan seluruh data train + early stopping
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)

start = time.perf_counter()
model_lgb = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=200,
    valid_sets=[dtrain, dval],
    callbacks=[lgb.early_stopping(30, verbose=False)]
)
time_lgb = time.perf_counter() - start

train_preds = model_lgb.predict(X_tr, num_iteration=model_lgb.best_iteration)
val_preds = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)

train_auc_lgb = roc_auc_score(y_tr, train_preds)
val_auc_lgb = roc_auc_score(y_va, val_preds)
print(f"Train AUC: {train_auc_lgb:.5f}")
print(f"Val AUC  : {val_auc_lgb:.5f}")
print(f"Gap      : {train_auc_lgb - val_auc_lgb:.5f}")
print(f"Time     : {time_lgb:.1f}s")

In [ ]:
# top 20 fitur berdasarkan gain importance
fi_lgb = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_lgb.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_lgb = fi_lgb.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_lgb["feature"].iloc[::-1], top20_lgb["importance"].iloc[::-1], color="#4c72b0")
ax.set_xlabel("Gain importance")
ax.set_title("LightGBM \u2014 Top 20 features by gain")
plt.tight_layout()
Path("../data/metadata").mkdir(parents=True, exist_ok=True)
plt.savefig("../data/metadata/lgbm_feature_importance_top20.png")
plt.show()

### 10.2 XGBoost

In [ ]:
# 5-fold cross validation dengan stratifikasi
xgb_scores = []
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "verbosity": 0,
    "random_state": 42
}

start = time.perf_counter()
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_va, label=y_va)
    
    early_stop = xgb.callback.EarlyStopping(rounds=30, save_best=True, metric_name="auc", data_name="val")
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=200,
        evals=[(dtrain, "train"), (dval, "val")],
        callbacks=[early_stop],
        verbose_eval=False
    )
    
    preds = model.predict(dval)
    xgb_scores.append(roc_auc_score(y_va, preds))

elapsed_xgb = time.perf_counter() - start
mean_xgb = np.mean(xgb_scores)
std_xgb = np.std(xgb_scores)
print(f"Mean AUC: {mean_xgb:.5f} \u00b1 {std_xgb:.5f}")
print(f"Fold scores: {[round(s, 5) for s in xgb_scores]}")

In [ ]:
# latih model final dengan seluruh data train + early stopping
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_va, label=y_va)
early_stop = xgb.callback.EarlyStopping(rounds=30, save_best=True, metric_name="auc", data_name="val")

start = time.perf_counter()
model_xgb = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dval, "val")],
    callbacks=[early_stop],
    verbose_eval=False
)
time_xgb = time.perf_counter() - start

train_preds = model_xgb.predict(dtrain)
val_preds = model_xgb.predict(dval)

train_auc_xgb = roc_auc_score(y_tr, train_preds)
val_auc_xgb = roc_auc_score(y_va, val_preds)
print(f"Train AUC: {train_auc_xgb:.5f}")
print(f"Val AUC  : {val_auc_xgb:.5f}")
print(f"Gap      : {train_auc_xgb - val_auc_xgb:.5f}")
print(f"Time     : {time_xgb:.1f}s")

In [ ]:
# feature importance
score_dict = model_xgb.get_score(importance_type="gain")
fi_xgb = pd.DataFrame({
    "feature": feature_cols,
    "importance": [score_dict.get(f, 0.0) for f in feature_cols]
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_xgb = fi_xgb.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_xgb["feature"].iloc[::-1], top20_xgb["importance"].iloc[::-1], color="#dd8452")
ax.set_xlabel("Gain importance")
ax.set_title("XGBoost \u2014 Top 20 features by gain")
plt.tight_layout()
plt.savefig("../data/metadata/xgb_feature_importance_top20.png")
plt.show()

### 10.3 CatBoost

In [ ]:
# 5-fold cross validation dengan stratifikasi
cb_scores = []
cb_params = {
    "iterations": 200,
    "learning_rate": 0.1,
    "depth": 6,
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False
}

start = time.perf_counter()
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, label=y_tr)
    val_pool = Pool(X_va, label=y_va)
    
    model = CatBoostClassifier(**cb_params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=30)
    
    preds = model.predict_proba(val_pool)[:, 1]
    cb_scores.append(roc_auc_score(y_va, preds))

elapsed_cb = time.perf_counter() - start
mean_cb = np.mean(cb_scores)
std_cb = np.std(cb_scores)
print(f"Mean AUC: {mean_cb:.5f} \u00b1 {std_cb:.5f}")
print(f"Fold scores: {[round(s, 5) for s in cb_scores]}")

In [ ]:
# latih model final dengan seluruh data train + early stopping
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
train_pool = Pool(X_tr, label=y_tr)
val_pool = Pool(X_va, label=y_va)

start = time.perf_counter()
model_cb = CatBoostClassifier(**cb_params)
model_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=30)
time_cb = time.perf_counter() - start

train_preds = model_cb.predict_proba(train_pool)[:, 1]
val_preds = model_cb.predict_proba(val_pool)[:, 1]

train_auc_cb = roc_auc_score(y_tr, train_preds)
val_auc_cb = roc_auc_score(y_va, val_preds)
print(f"Train AUC: {train_auc_cb:.5f}")
print(f"Val AUC  : {val_auc_cb:.5f}")
print(f"Gap      : {train_auc_cb - val_auc_cb:.5f}")
print(f"Time     : {time_cb:.1f}s")

In [ ]:
# feature importance
fi_cb = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_cb.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_cb = fi_cb.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_cb["feature"].iloc[::-1], top20_cb["importance"].iloc[::-1], color="#55a868")
ax.set_xlabel("Feature importance")
ax.set_title("CatBoost \u2014 Top 20 features")
plt.tight_layout()
plt.savefig("../data/metadata/cb_feature_importance_top20.png")
plt.show()

### 10.4 Cross-Validation Comparison

In [ ]:
cv_results = {
    "LightGBM": lgb_scores,
    "XGBoost": xgb_scores,
    "CatBoost": cb_scores
}
cv_df = pd.DataFrame(cv_results)
print(cv_df.describe().transpose())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=cv_df, ax=ax)
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Model Comparison \u2014 Cross-Validation Scores")
plt.show()

### 10.5 Full Training Comparison

In [ ]:
comparison = pd.DataFrame([
    {"model": "lightgbm", "val_auc": val_auc_lgb, "train_auc": train_auc_lgb, "gap": train_auc_lgb - val_auc_lgb, "time": time_lgb},
    {"model": "xgboost", "val_auc": val_auc_xgb, "train_auc": train_auc_xgb, "gap": train_auc_xgb - val_auc_xgb, "time": time_xgb},
    {"model": "catboost", "val_auc": val_auc_cb, "train_auc": train_auc_cb, "gap": train_auc_cb - val_auc_cb, "time": time_cb},
])
print(comparison.to_string(index=False))